In [ ]:
# install packages (only for the first time)
!pip install datasets pandas
!pip install -U datasets huggingface_hub fsspec

from datasets import load_dataset
import pandas as pd

# load GoEmotions dataset
goemotions = load_dataset("go_emotions", "simplified", split="train")

# convert it into a pandas DataFrame Pandas
df = goemotions.to_pandas()

# Keep samples with only one label
df["num_labels"] = df["labels"].apply(len)
df_single_label = df[df["num_labels"] == 1].copy()

# map label integers with emotions names
label_names = goemotions.features["labels"].feature.names
df_single_label["mood"] = df_single_label["labels"].apply(lambda x: label_names[x[0]])
df_single_label["caption"] = df_single_label["text"]

# select relevant features
final_df = df_single_label[["mood", "caption"]]

# save on csv file
final_df.to_csv("mood_captions_goemotions.csv", index=False)

# print some sample rows
print(final_df.sample(10))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/350k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

              mood                                            caption
9302   disapproval  You're willing to lie about anything to shill ...
19534      neutral  When it comes to cooking thermometers are your...
19476     approval  Ya see I’m an engineer that means I solve prac...
2233       neutral                      He’s Russian you forgot that.
33977      neutral  Anything other than harassing motorists or wor...
36809      neutral  A thousand year old institution that is hundre...
29883       desire  Bro, too close to home. Thing is I wish I coul...
2927      approval  Absolutely true. Sit down to pee at home, the ...
5545       neutral  I report these every time they appear on the b...
13037      neutral              Dems can end this. They chose not to.


In [ ]:
final_df

,mood,caption
0,neutral,My favourite food is anything I didn't have to...
1,neutral,"Now if he does off himself, everyone will thin..."
2,anger,WHY THE FUCK IS BAYLESS ISOING
3,fear,To make her feel threatened
4,annoyance,Dirty Southern Wankers
...,...,...
43405,love,Added you mate well I’ve just got the bow and ...
43406,confusion,Always thought that was funny but is it a refe...
43407,annoyance,What are you talking about? Anything bad that ...
43408,excitement,"More like a baptism, with sexy results!"


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch
import os
os.environ["WANDB_DISABLED"] = "true"

# load the previously created dataset from csv file
df = pd.read_csv("mood_captions_goemotions.csv")

# create a prompt like: "<mood>: <caption>"
df["text"] = df.apply(lambda row: f"mood: {row['mood']}\ncaption: {row['caption']}", axis=1)

# converti into HuggingFace dataset
dataset = Dataset.from_pandas(df[["text"]])

# Tokenizer & Model GPT-2
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # required for padding

model = GPT2LMHeadModel.from_pretrained(model_name)

# dataset tokenization
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# data collator for Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# training parameters
training_args = TrainingArguments(
    output_dir="./gpt2-mood-caption",
    report_to="none",
    # evaluation_strategy="no",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# start fine-tuning
trainer.train()

# save the fine-tuned model
trainer.save_model("./gpt2-mood-caption")
tokenizer.save_pretrained("./gpt2-mood-caption")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Map:   0%|          | 0/36308 [00:00<?, ? examples/s]

/tmp/ipython-input-3-573466890.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.333500
200,3.089600
300,3.088500
400,3.012700
500,2.953000
600,3.019600
700,2.993000
800,2.975200
900,2.995500
1000,2.958100


('./gpt2-mood-caption/tokenizer_config.json',
 './gpt2-mood-caption/special_tokens_map.json',
 './gpt2-mood-caption/vocab.json',
 './gpt2-mood-caption/merges.txt',
 './gpt2-mood-caption/added_tokens.json')

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="./gpt2-mood-caption", tokenizer="./gpt2-mood-caption")

prompt = "mood: joy"
output = generator(prompt, max_length=5, num_return_sequences=1, do_sample=True, temperature=0.5)
print(output[0]["generated_text"])


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=5) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


mood: joy
caption: I’m happy to help! I’m not a lawyer, but I am happy to share my thoughts. :) You’re welcome! ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ I hope you found this helpful. ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[NAME]~~ ~~[
